<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Feature Engineering - Solution

---

### About
Explain what feature engineering is, why it is important, and how to do it.

### Learning Objectives
- Explain the reasons for applying feature engineering to the data when modeling.
- Identify the relevant methods to apply to categorical data.
- Understand how and when to group continuous data into bins.
- Explain the reasons to transform your continuous data when modeling.
- Create transformers with scikit-learn to develop a pipeline of transformations for your data.

### Notebook Guide
1. What is Feature Engineering?
2. Working with Categorical Data
     - Ordinal Encoding
     - One-Hot Encoding
3. Binning Continuous Variables
4. Transforming Continuous Variables
5. Creating new features
6. Conclusions and Takeaways

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

# These are needed for the second part of the lesson (Transforming Continuous Variables and Pipelines)
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, KBinsDiscretizer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from sklearn import metrics


# What is Feature Engineering?

Feature engineering is the process of transforming raw data into features that better represent the underlying problem to the predictive models, resulting in improved model accuracy on unseen data.

Think of it as preparing your ingredients before cooking - you wouldn't throw whole vegetables into a soup without chopping them first!

## The Data

We'll use a housing data set that contains various features about houses and their sale prices. This is perfect for demonstrating different feature engineering techniques.

In [ ]:
# Load the housing data
# For this demo, we'll create a synthetic data set that mimics real housing data
np.random.seed(42)

n_samples = 1000

# Create synthetic housing data
housing_data = {
    'sqft': np.random.normal(2000, 500, n_samples),
    'bedrooms': np.random.choice([1, 2, 3, 4, 5], n_samples, p=[0.1, 0.2, 0.4, 0.25, 0.05]),
    'bathrooms': np.random.choice([1, 1.5, 2, 2.5, 3, 3.5, 4], n_samples, p=[0.15, 0.1, 0.3, 0.15, 0.2, 0.05, 0.05]),
    'neighborhood': np.random.choice(['Downtown', 'Suburb', 'Rural', 'Waterfront'], n_samples, p=[0.3, 0.5, 0.15, 0.05]),
    'year_built': np.random.randint(1950, 2023, n_samples),
    'garage': np.random.choice(['None', 'Attached', 'Detached'], n_samples, p=[0.2, 0.6, 0.2]),
    'condition': np.random.choice(['Poor', 'Fair', 'Good', 'Excellent'], n_samples, p=[0.1, 0.2, 0.5, 0.2]),
    'income_area': np.random.exponential(50000, n_samples) + 30000
}

housing = pd.DataFrame(housing_data)

# Make sqft positive
housing['sqft'] = np.abs(housing['sqft'])

# Create a realistic price based on features
price_base = (housing['sqft'] * 150 + 
              housing['bedrooms'] * 10000 + 
              housing['bathrooms'] * 8000 + 
              (2023 - housing['year_built']) * -500 +
              housing['income_area'] * 2)

# Add neighborhood effects
neighborhood_effects = {'Downtown': 50000, 'Waterfront': 80000, 'Suburb': 20000, 'Rural': -10000}
for neighborhood, effect in neighborhood_effects.items():
    price_base += (housing['neighborhood'] == neighborhood) * effect

# Add some noise
housing['price'] = price_base + np.random.normal(0, 20000, n_samples)
housing['price'] = np.abs(housing['price'])  # Ensure positive prices

In [ ]:
housing.head()

In [ ]:
housing.describe()

Let's take a quick look at our data types and see what we're working with:

In [ ]:
housing.info()

# Intro to Sklearn's Transformers

We first load in the package like so:

```python
from sklearn.preprocessing import transformer
```

Once instantiated, the transformer object has three primary methods built in:
- `.fit(X)` will calculate statistical parameters for each column of X
- `.transform(X)` will take X and return a transformed version of X where each column is modified according to the learned parameters.
- `.fit_transform(X)` combines the `.fit()` method and the `.transform()`method.

Examples:

```python
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.preprocessing import PowerTransformer, QuantileTransformer, KBinsDiscretizer
```


---

# Working with Categorical Data

Categorical variables are variables that take on a limited number of distinct values. Most machine learning algorithms can't handle categorical data directly - they need numbers!

## Identifying Categorical Variables

Let's identify the unique values in each column for all features in our data set that are categorical:

In [ ]:
# Look at unique values in categorical columns
categorical_cols = housing.select_dtypes(include="object").columns
categorical_cols

In [ ]:
# Display the different values of each categorical feature
for col in categorical_cols:
    print(f"{col}: {housing[col].unique()}")

<details><summary>What are the ordinal features? Nominal?</summary>
    
- **Ordinal features**: neighborhood, garage
  
- **Nominal features**: condition
</details>



## Ordinal Encoding

Ordinal encoding converts ordered categories into numbers that preserve their natural order, e.g. `Low=0`, `Medium=1`, `High=2`.

In [ ]:
# Let's ordinal encode the 'condition' variable since it has a natural order

# Step 1. Instantiate the transformer
o_enc = OrdinalEncoder( categories=[ ["Poor","Fair","Good", "Excellent"] ]).set_output(transform = "pandas")

# Step 2. Fit the transformer on your feature
o_enc.fit(housing[["condition"]])

# during fit, the encoder learns the different categories in features with their ranking
# during transform, the data is labelled 

In [ ]:
# Step 3. Transform the feature

ordinal_enc = o_enc.transform( housing[["condition"]] )
ordinal_enc.head()

<details><summary>Why is ordinal encoding appropriate for 'condition' but not for 'neighborhood'?</summary>
Condition has a natural order (Poor < Fair < Good < Excellent), while neighborhood categories don't have a meaningful order.
</details>

## One-Hot Encoding

One-hot encoding creates binary columns for each category. This is better for nominal data (no natural order).

In [ ]:
# One-hot encode the neighborhood variable

# Step 1. Instantiate the transformer
ohe = OneHotEncoder(
                drop="first", #drops first column
                handle_unknown='ignore', # encodes an unknown category as all zeros
                sparse_output=False,  #to not compress the results
                    ).set_output(transform = "pandas")

# Step 2 & 3. Fit and transform
nominal_enc = ohe.fit_transform(housing[['neighborhood','garage']])
nominal_enc.head()

# during fit, the encoder learns the different categories in features
# during transform, the data is labelled 

<details><summary>Why we drop the first column?</summary>
    
To avoid **Perfect Multicollinearity**:
    
- Happens when one feature can be calculated exactly from other features.
- Common with one-hot encoding if you keep all dummy columns.
- Example:
    - Red = 1 - Blue - Green
- The dropped category becomes the reference group.

</details>


In [ ]:
#TO_DO: Apply an encoding for the df below

df = pd.DataFrame({
    "service_package": [
        "Select",
        "Essential",
        "Preferred",
        "Elite",
        "Select",
        "Preferred",
        "Essential",
        "Elite",
        "Preferred",
        "Select",
        "Essential",
        "Preferred",
        "Elite",
        "Select",
        "Select",
        "Essential",
        "Elite",
        "Preferred",
        "Essential",
        "Preferred"
    ]
})

df.head()

In [ ]:
# Solution



---

#### To the slides!

---


# Binning Continuous Variables

Sometimes it's useful to convert continuous variables into categorical bins. This can help capture non-linear relationships and make models more interpretable.

## When to Bin Continuous Variables

- When you suspect non-linear relationships
- To reduce the impact of outliers
- To make model interpretations easier
- When domain knowledge suggests natural breakpoints

## Manual Binning with `pandas.cut()`

Let's bin the house `age` into categories:

In [ ]:
# Create an age variable first
housing['age'] = datetime.now().year - housing['year_built']
print(f"Age range: {housing['age'].min()} to {housing['age'].max()}")

In [ ]:
# Show the age distribution
sns.histplot(housing['age'])
plt.title("Age Distribution");

In [ ]:
# Show the relation between age and target

sns.scatterplot(x = housing['age'], y=housing["price"] )
plt.title("Age vs Price");

In [ ]:
# Create age bins


# Define the boundaries for each age group
# This creates the ranges: 0–10, 10–20, 20–30, 30–50, 50–100
age_bins = [0, 10, 20, 30, 50, 100]

# Define a label for each age range
age_labels = ['Very New', 'New', 'Moderate', 'Old', 'Very Old']

# Convert the numerical 'age' feature into categorical age groups
housing['age_category'] = pd.cut(
                            housing['age'],          # Column to bin
                            bins=age_bins,           # Bin boundaries
                            labels=age_labels,       # Labels assigned to each bin
                            include_lowest=True      # Include the minimum value in the first bin
                                )

In [ ]:
print("Age category distribution:")
housing['age_category'].value_counts()

In [ ]:
# Plot average house price by income category

housing.groupby('age_category')['price'].mean().plot(kind="bar")

plt.title('Average House Price by Age Category')
plt.xlabel('Income Level')
plt.ylabel('Age')
plt.xticks(rotation=0)
plt.show();

> **DONT FORGET TO ENCODE THE NEW FEATURE**

<details><summary>Do you think this new feature is useful?</summary>
        
Average price barely changes across the labels. Age doesn't seem to have a strong predictive power

</details>

## Equal-Frequency (quantile-based) Binning with `pandas.qcut()`

Sometimes we want equal-sized bins rather than equal-width or fixed-width bins:
- It creates groups based on quantiles / percentiles, not equal numeric widths.
- Useful for skewed variables, where equal/fixed-width bins may leave some groups nearly empty.
- Useful when relative position matters, e.g. bottom 25%, middle groups, top 25%
    - **Customer spending**: you may care about the top 20% of customers by spend for a loyalty campaign, regardless of whether the cutoff happens to be `500` or `900`.
    - **Sales performance**: identify the top 25% of salespeople rather than defining an arbitrary sales threshold.
- Use it when there are no meaningful domain-based cutoffs and you still want balanced groups.

In [ ]:
# Show the income distribution
sns.histplot( housing['income_area'])
plt.title("Income Area Distribution");

In [ ]:
# Use KBinsDiscretizer on income

binned = KBinsDiscretizer(
                            n_bins=4,  #split the feature into 4 groups
                            encode='ordinal', #label the groups as 0, 1, 2, 3
                            strategy='quantile' #each group contains roughly 25% of the observations
                         ).set_output(transform = "pandas")

# Fit and transform
income_binned = binned.fit_transform(housing[['income_area']])
income_binned.head()

# Here .fit() learns the cutoff values and transform() uses those cutoffs to assign each row to a bin.

In [ ]:
print("Income quartile distribution:")
income_binned.value_counts()

In [ ]:
# Plot income vs price by quartile
housing["income_binned"] = income_binned.values
housing.groupby('income_binned')['price'].mean().plot(kind="bar")

plt.title('Average House Price by Income Quartile')
plt.xlabel('Income Quartile')
plt.ylabel('Average Price')
plt.xticks(rotation=0);

<details><summary>Do you think this new feature is useful?</summary>
        
Average price  changes across the labels. Income does seem to have a strong predictive power

</details>

In [ ]:
#TO-DO: Choose the right binning strategy for the df below.

#Steps:
#1. Show distribution
#2. Apply binning strategy
#3. Display the group size

customers = pd.DataFrame({
    "annual_spend": [
        120, 150, 175, 190, 210,
        225, 240, 255, 270, 285,
        300, 315, 330, 345, 360,
        380, 400, 420, 440, 460,
        480, 500, 525, 550, 580,
        610, 650, 700, 760, 820,
        900, 1000, 1150, 1300, 1500,
        1800, 2200, 2800, 3600, 5000
    ]
})

customers.head()


In [ ]:
# Solution



In [ ]:
# 2. Binning


In [ ]:
# 3. Group sizes


---

#### To the slides!

---

# Transforming Continuous Variables

Many machine learning algorithms work better when features are on similar scales or follow certain distributions.

## Why Scale?

- **Prevent large-scale features from dominating** models that depend on magnitudes.
- **Make distance calculations meaningful** for models such as KNN and SVM.
- **Improve optimization and convergence** for models trained iteratively, such as neural networks and regularized linear/logistic regression.
- **Put features on a comparable scale**



## Visualizing Our Continuous Variables

Let's look at the distributions first:

In [ ]:
# Plot distributions of continuous variables
continuous_vars = ['sqft', 'price']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for i, var in enumerate(continuous_vars):
    sns.histplot(housing[var], bins=30, alpha=0.7, ax= axes[i])
    axes[i].set_title(f'Distribution of {var}')
    axes[i].set_xlabel(var)
    axes[i].set_ylabel('Frequency')

<details><summary>Which scaling technique would you select for "sqft?"</summary>
        
Standard scaler because:
- distribution is normal
- no outliars
- no natural bounds
</details>

## Scaling Transformations

### StandardScaler (Z-score normalization)
Transforms data to have mean=0 and std=1

In [ ]:
# Apply StandardScaler to sqft
scaler = StandardScaler().set_output(transform = "pandas")

sqrt_scaled = scaler.fit_transform(housing[['sqft']])
# fit learns the mean and standard deviation of the feature

sqrt_scaled.head()

In [ ]:
print("Original sqft stats:")
print(f"Mean: {housing['sqft'].mean():.2f}")
print(f"Std: {housing['sqft'].std():.2f}")

print("\nScaled sqft stats:")
print(f"Mean: {sqrt_scaled['sqft'].mean():.2f}")
print(f"Std: {sqrt_scaled['sqft'].std():.2f}")

In [ ]:
# Plot before and after scaling
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(housing['sqft'], bins=30, alpha=0.7, ax = axs[0])
axs[0].set_title('Original Square Footage')
axs[0].set_xlabel('Square Footage')

sns.histplot(sqrt_scaled['sqft'], bins=30, alpha=0.7, ax=axs[1])
axs[1].set_title('Standardized Square Footage')
axs[1].set_xlabel('Standardized Square Footage');


### MinMaxScaler
Scales data to a fixed range (usually 0-1)

In [ ]:
# Practice: Apply MinMaxScaler to income_area
minmax_scaler = MinMaxScaler().set_output(transform = "pandas")

income_scaled = minmax_scaler.fit_transform(housing[['income_area']])
# fit learns the Max and Min Values of the features

income_scaled.head()

In [ ]:
print("Original income_area stats:")
print(f"Min: {housing['income_area'].min():.2f}")
print(f"Max: {housing['income_area'].max():.2f}")

print("\nScaled income_area stats:")
print(f"Min: {income_scaled['income_area'].min():.2f}")
print(f"Max: {income_scaled['income_area'].max():.2f}")

In [ ]:
# Plot before and after MinMax scaling
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(housing['income_area'], bins=30, alpha=0.7, ax= axs[0])
axs[0].set_title('Original Income Area')
axs[0].set_xlabel('Income Area')

sns.histplot(income_scaled['income_area'], bins=30, alpha=0.7, ax=axs[1])
axs[1].set_title('MinMax Scaled Income Area')
axs[1].set_xlabel('Scaled Income Area');

In [ ]:
#TO_DO: Choose the right scale technique for the dataframe below and apply it.

df = pd.DataFrame({
    "monthly_spend": [
        410, 425, 440, 455, 470,
        480, 490, 500, 510, 520,
        530, 540, 550, 560, 570,
        580, 590, 600, 610, 620,
        630, 640, 650, 660, 670,
        680, 690, 700, 710, 720,
        730, 740, 750, 760, 770,
        780, 790, 800, 820, 840,
        860, 880, 900, 950, 1000,
        1200, 1500, 3000, 6000, 10000
    ]
})

df.head()

In [ ]:
# SOLUTION


## Distribution Transformations



### What is is?
Applying a mathematical function to a numerical variable to change its distribution shape or scale in a way that may improve modeling.

### Why we use it?

Many machine learning models perform poorly when features are highly skewed (e.g., Income, Population, Call Durations). 

* **Non-Normal Distributions:** Linear models and parametric tests assume that residuals are **normally distributed** with a **constant variance**. Highly skewed data violates this assumption, distorting confidence intervals and model predictions.
* **Sensitivity to Outliers:** Extreme values pull the mean and variance, causing models to prioritize fitting anomalies rather than the underlying trends.
_Check the effect of outliers on your Linear Regression model using this [link](https://seeing-theory.brown.edu/regression-analysis/)._

>_In other words, the transformation enhances the linerarity relationship between features and target_

### Log Transformation
Replaces each value with its natural or base-10 logarithm ($X \rightarrow \ln(X)$).
* **When to use:** Right-skewed (positive) distributions.
* **Effects:** Compresses large values and expand smaller ones
* **Limitation:** Undefined for zero or negative values. To handle zero, data scientists apply the $log(x + 1)$ transformation (implemented via `np.log1p`).



In [ ]:
# Let's take 'incomce_area' as a case scenario

sns.histplot( housing['income_area'])
plt.title("Income Area Distribution");

In [ ]:
# Apply log transformation to income_area (which is right-skewed)
housing['income_area_log'] = np.log(housing['income_area'])

print("Original income_area skewness:", housing['income_area'].skew())
print("Log-transformed income_area skewness:", housing['income_area_log'].skew())

#### What is Skewness?

**Skewness** is a statistical measure that quantifies the asymmetry of a data distribution around its mean. In a perfectly symmetrical distribution (like a standard normal bell curve), the skewness score is exactly **0**, and the mean, median, and mode are all equal.

#### Skewness Score Interpretation

| Skewness Score Range | Distribution Shape | 
| :--- | :--- | 
| **Less than -1** | Highly Skewed (Left) | 
| **Between -1 and -0.5** | Moderately Skewed (Left) | 
| **Between -0.5 and 0.5** | `Normal (Symmetrical)` | 
| **Between 0.5 and 1** | Moderately Skewed (Right) | 
| **Greater than 1** | Highly Skewed (Right) | 

In [ ]:
# Plot before and after log transformation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(housing['income_area'], bins=30, alpha=0.7, ax= ax1)
ax1.set_title('Original Income Area (Right-skewed)')
ax1.set_xlabel('Income Area')
ax1.set_ylabel('Frequency')

sns.histplot(housing['income_area_log'], bins=30, alpha=0.7, ax=ax2)
ax2.set_title('Log-transformed Income Area')
ax2.set_xlabel('Log(Income Area)')
ax2.set_ylabel('Frequency');


### `PowerTransformer` (Box-Cox and Yeo-Johnson)


#### What is Power Transformation?
A power transformation applies a mathematical power to numerical data to reduce skewness and make its distribution more suitable for modeling.

#### What is lambda?
- Lambda (λ) is the parameter that controls the strength and type of power transformation applied to the data.

- Transformer chooses the value that makes the transformed data most compatible with a normal distribution.

#### Box-Cox vs Yeo-Johnson

* **Box-Cox:** Extremely effective but strictly requires all input values to be **strictly positive** ($X > 0$).
* **Yeo-Johnson:** A modern modification of Box-Cox that allows for **zero and negative values**.

In [ ]:
# Use PowerTransformer on income_area
power_transformer = PowerTransformer(method='yeo-johnson').set_output(transform="pandas")

income_power = power_transformer.fit_transform(housing[['income_area']])
#Here the transformer calculates lambda based on the feature values

In [ ]:
print("Original income_area skewness:", housing['income_area'].skew())
print("Power-transformed income_area skewness:", income_power['income_area'].skew())

In [ ]:
# Plot before and after power transformation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(housing['income_area'], bins=30, alpha=0.7, ax=ax1)
ax1.set_title('Original Income Area')
ax1.set_xlabel('Income Area')

sns.histplot(income_power['income_area'], bins=30, alpha=0.7, ax=ax2)
ax2.set_title('Power-transformed Income Area')
ax2.set_xlabel('Power-transformed Income Area');

### STRETCH: `QuantileTransformer`
Maps the original feature distribution to a target distribution (typically a uniform or normal distribution) using non-parametric rank transformation.
* **When to use:** Highly non-linear, bimodal, or multi-modal distributions that parametric transformations fail to smooth out.
* **Limitation:** Distorts the absolute linear distances between close values; breaks down standard correlation metrics.

In [ ]:
# Use QuantileTransformer
quantile_transformer = QuantileTransformer(output_distribution='normal').set_output(transform="pandas")

income_quantile = quantile_transformer.fit_transform(housing[['income_area']])

print("Original income_area skewness:", housing['income_area'].skew())
print("Quantile-transformed income_area skewness:", income_quantile['income_area'].skew())

### Decision Matrix: Choosing the Right Distribution Transformation

| Transformation | Best Data Shape / Scenario | Handles Zeros? | Handles Negatives? | Impact on Linear Models / Interpretability |
| :--- | :--- | :--- | :--- | :--- |
| **Log Plus One ($\log(x+1)$)** | Highly Right-Skewed with zero counts |  Yes | ❌ No | Safely maps zero values to zero while retaining standard log-compression benefits. |
| **Box-Cox** | Right or Left Skewed (Strictly positive) | ❌ No | ❌ No | Automatically searches for the mathematically ideal power exponent ($\lambda$) to maximize normality. |
| **Yeo-Johnson** | Right or Left Skewed (Versatile) |  Yes |  Yes | Modifies the Box-Cox algorithm to safely calculate ideal power scales across zeros and negatives. |
| **Quantile Transformer** | Multi-modal, heavily skewed, or extreme outliers |  Yes |  Yes | Forces any distribution pattern into a smooth, normal shape. **Warning:** Not to be used with Linear Regression. It distorts relative variable distance. |

In [ ]:
# Compare all transformations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

# Adding transformations to dataframe

housing['income_area_power'] = income_power.values
housing['income_area_quantile'] = income_quantile.values

transformations = [
    ('Original', 'income_area'),
    ('Log', 'income_area_log'),
    ('Power', 'income_area_power'),
    ('Quantile', 'income_area_quantile')
]

for i, (name, col) in enumerate(transformations):
    axes[i].hist(housing[col], bins=30, alpha=0.7)
    axes[i].set_title(f'{name} Transformation')
    axes[i].set_xlabel(col)
    
    # Add skewness to title
    skew = housing[col].skew()
    axes[i].set_title(f'{name} Transformation (skew: {skew:.2f})')

plt.tight_layout()
plt.show();

In [ ]:
#TO-DO: Choose the right power transformation for the dataset below
# Show the distribution before and after applying the transformation

df = pd.DataFrame({
    "profit_margin_change": [
        -52, -45, -40, -36, -32, -29, -26, -24, -22, -20,
        -18, -17, -16, -15, -14, -13, -12, -11, -10, -9,
        -8, -8, -7, -7, -6, -6, -5, -5, -4, -4,
        -3, -3, -3, -2, -2, -2, -1, -1, -1, 0,
        0, 0, 1, 1, 1, 2, 2, 2, 3, 3,
        3, 4, 4, 4, 5, 5, 5, 6, 6, 7,
        7, 8, 8, 9, 9, 10, 10, 11, 12, 12,
        13, 14, 15, 16, 17, 18, 20, 22, 24, 26,
        28, 30, 33, 36, 40, 45, 50, 56, 63, 70,
        80, 92, 105, 120, 140, 165, 200, 250, 330, 450
    ]
})

df.head()

In [ ]:
#SOLUTION:




# Combining It All
Let's create a final df containing all our preprocessed data

In [ ]:
#Below are all the dfs holding our chosen transformations
ordinal_enc
nominal_enc
income_binned
sqrt_scaled


# Concatinate all transformations into a single dataframe
X_prep = pd.concat([sqrt_scaled,income_binned,nominal_enc, ordinal_enc], axis=1)
X_prep.head()

# Preprocessing Before or After Splitting Data?
- Split the data before any preprocessing step that learns from the dataset to prevent information from the test set leaking into training.
- The training set should be the only data used to learn transformation parameters ( mean, standard deviation, lambda values, category statistics,...)
- After fitting preprocessing on the training set, apply the same learned transformation to the test set.
- This keeps the test set truly unseen and gives a more realistic estimate of model performance.
- **Not every transformation has to happen after the split**:
    - Fixed transformations that do not learn from the dataset—such as log1p(x), ratios, arithmetic features, or predefined domain bins can be created before splitting.

In [ ]:
# Sample workflow

# 1. log transform income before split
housing["income_log"] = np.log(housing["income_area"])

# 2. Choose X and y (select only sqft and income_log as features)
X = housing[["sqft", "income_log","neighborhood"]]
y = housing["price"]

# 2. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.25, random_state=42)

# 3. Fit transform on train set
# 3.1 Apply scaling on numerical features
X_train_num = X_train[["sqft", "income_log"]]
scaler = StandardScaler().set_output(transform = "pandas")
X_train_scaled = scaler.fit_transform(X_train_num)

# 3.2 Encode categorical features
X_train_cat = X_train[["neighborhood"]]
ohe = OneHotEncoder(
                drop="first", #drops first column
                handle_unknown='ignore', # encodes an unknown category as all zeros
                sparse_output=False,  #to not compress the results
                    ).set_output(transform = "pandas")
X_train_enc = ohe.fit_transform(X_train_cat)


# 4. Transform on test set
X_test_num = X_test[["sqft", "income_log"]]
X_test_cat = X_test[["neighborhood"]]

X_test_scaled = scaler.transform( X_test_num )
X_test_enc= ohe.transform( X_test_cat )

# 5. Concatinate the transformations

X_train_prep = pd.concat([X_train_scaled,X_train_enc], axis=1)
X_test_prep = pd.concat([X_test_scaled,X_test_enc], axis=1)

# 6. Fit model
model = LinearRegression ()
model.fit(X_train_prep, y_train)

# 7. Evaluate Model
model.score(X_test_prep,y_test)


# Conclusions and Takeaways

* **Feature engineering is crucial** for model performance - sometimes more important than the choice of algorithm!
* **Different data types need different treatments**: categorical vs. continuous variables require different encoding strategies.
* **Scaling matters** for many algorithms, especially distance-based ones.
* **Domain knowledge** often guides the best feature engineering choices.
* **Always validate** your transformations - plot before and after!

### Next Steps

Practice these techniques on your own data sets! Remember:
- Start simple and add complexity gradually
- Always split your data before engineering features
- Document your transformations for reproducibility
- Experiment with different approaches and measure their impact